# SaaS Analytics Dashboard

End-to-end analysis of synthetic SaaS data (seed=42, Jan 2023–Dec 2025). Source: `../data/saas_data.db` (SQLite).

**Sections:** Data Overview · MRR Waterfall · Cohort Retention · Retention by Channel + Anomaly · LTV/CAC

In [2]:
import sqlite3
import csv
from pathlib import Path
from collections import defaultdict

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CWD = Path.cwd().resolve()
BASE_DIR = CWD if (CWD / 'data').exists() else CWD.parent
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs'
DB_PATH = str(DATA_DIR / 'saas_data.db')

CHANNEL_COLORS = {
    'organic_search':    '#2c3e50',
    'paid_search':       '#2980b9',
    'partner_referral':  '#e74c3c',
    'content_marketing': '#27ae60',
    'direct':            '#8e44ad',
    'social_media':      '#f39c12',
}

PLAN_COLORS = {
    'Starter':    '#5b8dd9',
    'Growth':     '#f0a500',
    'Enterprise': '#e05c5c',
}

## 1. Data Overview

Row counts and key distributions across the four tables in `../data/saas_data.db`.

In [3]:
conn = sqlite3.connect(DB_PATH)
print('Table row counts:')
for table in ['dim_users', 'fact_subscriptions', 'fact_payments', 'fact_events']:
    n = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'  {table:25s}: {n:>12,}')

print('\nPlan distribution (fact_subscriptions):')
for plan, cnt in conn.execute(
    'SELECT plan, COUNT(*) FROM fact_subscriptions GROUP BY plan ORDER BY plan'
).fetchall():
    print(f'  {plan:12s}: {cnt:>6,}')

print('\nChannel distribution (dim_users):')
for ch, cnt in conn.execute(
    'SELECT acquisition_channel, COUNT(*) FROM dim_users '
    'GROUP BY acquisition_channel ORDER BY acquisition_channel'
).fetchall():
    print(f'  {ch:22s}: {cnt:>6,}')

print('\nDate range (fact_subscriptions):')
row = conn.execute('SELECT MIN(started_at), MAX(started_at) FROM fact_subscriptions').fetchone()
print(f'  {row[0]} to {row[1]}')
conn.close()

Table row counts:
  dim_users                :       10,000
  fact_subscriptions       :       13,066
  fact_payments            :      153,772
  fact_events              :   25,881,093

Plan distribution (fact_subscriptions):
  Enterprise  :  1,868
  Growth      :  4,659
  Starter     :  6,539

Channel distribution (dim_users):
  content_marketing     :  1,509
  direct                :    990
  organic_search        :  3,066
  paid_search           :  1,938
  partner_referral      :  1,780
  social_media          :    717

Date range (fact_subscriptions):
  2023-02-01 to 2025-12-28


## 2. MRR Movement

Classifies every customer-month MRR change into: **New | Expansion | Contraction | Churn | Reactivation | Retained**.

- Recursive CTE expands subscriptions into monthly rows
- `LAG()` detects movement type by comparing current vs prior-month MRR
- Separate `LEAD()` query detects churn (active last month, absent next month)

In [4]:
CUSTOMER_MONTH_MRR_SQL = '''
WITH RECURSIVE months AS (
    SELECT '2023-01-01' AS month_start
    UNION ALL
    SELECT date(month_start, '+1 month') FROM months
    WHERE month_start < '2025-12-01'
),
sub_months AS (
    SELECT
        s.user_id,
        s.subscription_id,
        m.month_start,
        s.mrr,
        s.plan
    FROM fact_subscriptions s
    JOIN months m
      ON m.month_start >= strftime('%Y-%m-01', s.started_at)
     AND (s.ended_at IS NULL OR m.month_start < strftime('%Y-%m-01', s.ended_at, '+1 month'))
),
customer_month AS (
    SELECT
        user_id,
        month_start,
        MAX(mrr) AS mrr,
        (SELECT sm2.plan FROM sub_months sm2
         WHERE sm2.user_id = sub_months.user_id
           AND sm2.month_start = sub_months.month_start
         ORDER BY sm2.mrr DESC, sm2.subscription_id DESC LIMIT 1) AS plan
    FROM sub_months
    GROUP BY user_id, month_start
),
with_prior AS (
    SELECT
        user_id, month_start, mrr, plan,
        LAG(mrr)         OVER (PARTITION BY user_id ORDER BY month_start) AS prev_mrr,
        LAG(month_start) OVER (PARTITION BY user_id ORDER BY month_start) AS prev_month
    FROM customer_month
)
SELECT
    user_id, month_start, mrr, prev_mrr, prev_month,
    CASE
        WHEN prev_mrr IS NULL THEN 'new'
        WHEN date(prev_month, '+1 month') < month_start THEN 'reactivation'
        WHEN mrr > prev_mrr THEN 'expansion'
        WHEN mrr < prev_mrr THEN 'contraction'
        ELSE 'retained'
    END AS movement_type,
    CASE
        WHEN prev_mrr IS NULL THEN mrr
        WHEN date(prev_month, '+1 month') < month_start THEN mrr
        ELSE mrr - prev_mrr
    END AS mrr_delta
FROM with_prior
ORDER BY month_start, user_id;
'''

CHURN_SQL = '''
WITH RECURSIVE months AS (
    SELECT '2023-01-01' AS month_start
    UNION ALL
    SELECT date(month_start, '+1 month') FROM months
    WHERE month_start < '2025-12-01'
),
sub_months AS (
    SELECT s.user_id, m.month_start, s.mrr
    FROM fact_subscriptions s
    JOIN months m
      ON m.month_start >= strftime('%Y-%m-01', s.started_at)
     AND (s.ended_at IS NULL OR m.month_start < strftime('%Y-%m-01', s.ended_at, '+1 month'))
),
customer_month AS (
    SELECT user_id, month_start, MAX(mrr) AS mrr
    FROM sub_months GROUP BY user_id, month_start
),
with_next AS (
    SELECT cm.user_id, cm.month_start, cm.mrr,
           LEAD(cm.month_start) OVER (PARTITION BY cm.user_id ORDER BY cm.month_start) AS next_month
    FROM customer_month cm
)
SELECT
    date(month_start, '+1 month') AS churn_month,
    user_id,
    -mrr AS mrr_delta
FROM with_next
WHERE next_month IS NULL OR date(month_start, '+1 month') < next_month;
'''

In [5]:
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA journal_mode=WAL')
cur = conn.cursor()

print('Classifying MRR movements per customer-month...')
movements = cur.execute(CUSTOMER_MONTH_MRR_SQL).fetchall()
print(f'  {len(movements):,} customer-month rows (non-churn)')

print('Detecting churn months...')
churns = cur.execute(CHURN_SQL).fetchall()
print(f'  {len(churns):,} churn events')
conn.close()

waterfall = defaultdict(lambda: {
    'new': 0.0, 'expansion': 0.0, 'reactivation': 0.0,
    'contraction': 0.0, 'churn': 0.0, 'retained_mrr': 0.0,
    'retained_count': 0,
})

for row in movements:
    month, movement, delta = row[1], row[5], row[6]
    if movement == 'retained':
        waterfall[month]['retained_mrr'] += row[2]
        waterfall[month]['retained_count'] += 1
    else:
        waterfall[month][movement] += delta

for churn_month, user_id, mrr_delta in churns:
    if churn_month <= '2025-12-01':
        waterfall[churn_month]['churn'] += mrr_delta

months_sorted = sorted(waterfall.keys())
cumulative_mrr = 0.0
mrr_rows = []
for month in months_sorted:
    w = waterfall[month]
    net = w['new'] + w['expansion'] + w['reactivation'] + w['contraction'] + w['churn']
    cumulative_mrr += net
    mrr_rows.append({
        'month': month,
        'beginning_mrr': round(cumulative_mrr - net, 2),
        'new': round(w['new'], 2),
        'expansion': round(w['expansion'], 2),
        'reactivation': round(w['reactivation'], 2),
        'contraction': round(w['contraction'], 2),
        'churn': round(w['churn'], 2),
        'net_new_mrr': round(net, 2),
        'ending_mrr': round(cumulative_mrr, 2),
        'retained_count': w['retained_count'],
    })
print(f'Aggregated {len(mrr_rows)} monthly waterfall rows')

Classifying MRR movements per customer-month...
  158,011 customer-month rows (non-churn)
Detecting churn months...
  11,244 churn events
Aggregated 35 monthly waterfall rows


In [6]:
fieldnames_mrr = list(mrr_rows[0].keys())
with open(DATA_DIR / 'mrr_waterfall.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames_mrr)
    writer.writeheader()
    writer.writerows(mrr_rows)
print(f'Wrote {len(mrr_rows)} months to mrr_waterfall.csv')

print(f"\n{'Month':>10s} {'Begin':>10s} {'New':>9s} {'Expan':>9s} "
      f"{'React':>9s} {'Contr':>9s} {'Churn':>10s} {'Net':>10s} {'Ending':>11s}")
for r in mrr_rows[:6] + [mrr_rows[-1]]:
    print(f"{r['month']:>10s} {r['beginning_mrr']:>10,.0f} {r['new']:>9,.0f} "
          f"{r['expansion']:>9,.0f} {r['reactivation']:>9,.0f} "
          f"{r['contraction']:>9,.0f} {r['churn']:>10,.0f} "
          f"{r['net_new_mrr']:>10,.0f} {r['ending_mrr']:>11,.0f}")
if len(mrr_rows) > 7:
    print(f'  ... ({len(mrr_rows) - 7} more rows in mrr_waterfall.csv)')

Wrote 35 months to mrr_waterfall.csv

     Month      Begin       New     Expan     React     Contr      Churn        Net      Ending
2023-02-01          0       555         0         0         0          0        555         555
2023-03-01        555     3,002         0         0         0          0      3,002       3,557
2023-04-01      3,557     9,083         0         0         0          0      9,083      12,640
2023-05-01     12,640    16,701       237         0         0       -128     16,810      29,450
2023-06-01     29,450    24,233       457         0         0       -430     24,259      53,709
2023-07-01     53,709    35,819       575         0       -70     -1,157     35,167      88,875
2025-12-01    620,766     4,325     5,850     6,896    -1,723    -19,055     -3,707     617,059
  ... (28 more rows in mrr_waterfall.csv)


In [7]:
months_labels = [r['month'][:7] for r in mrr_rows]
fig_waterfall = go.Figure()
fig_waterfall.add_trace(go.Bar(name='New',          x=months_labels,
                               y=[r['new'] for r in mrr_rows],         marker_color='#2ecc71'))
fig_waterfall.add_trace(go.Bar(name='Expansion',    x=months_labels,
                               y=[r['expansion'] for r in mrr_rows],    marker_color='#3498db'))
fig_waterfall.add_trace(go.Bar(name='Reactivation', x=months_labels,
                               y=[r['reactivation'] for r in mrr_rows], marker_color='#9b59b6'))
fig_waterfall.add_trace(go.Bar(name='Contraction',  x=months_labels,
                               y=[r['contraction'] for r in mrr_rows],  marker_color='#e67e22'))
fig_waterfall.add_trace(go.Bar(name='Churn',        x=months_labels,
                               y=[r['churn'] for r in mrr_rows],        marker_color='#e74c3c'))
fig_waterfall.add_trace(go.Scatter(
    name='Ending MRR', x=months_labels, y=[r['ending_mrr'] for r in mrr_rows],
    mode='lines+markers', line=dict(color='#2c3e50', width=2.5),
    marker=dict(size=5), yaxis='y2',
))
fig_waterfall.update_layout(
    title='MRR Waterfall: Monthly Movements (Jan 2023 - Dec 2025)',
    barmode='relative',
    yaxis=dict(title='MRR Movement ($)', tickformat='$,.0f'),
    yaxis2=dict(title='Cumulative MRR ($)', overlaying='y', side='right',
                tickformat='$,.0f', showgrid=False),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    template='plotly_white', height=550,
    margin=dict(l=80, r=80, t=80, b=60),
)
fig_waterfall.write_html(OUTPUT_DIR / 'mrr_waterfall.html', include_plotlyjs='cdn')
fig_waterfall.show()
fig_waterfall.write_image(OUTPUT_DIR / 'fig_waterfall.png')


### Static Image Preview\n
\n
![](../outputs/fig_waterfall.png)\n
\n

## 3. Cohort Retention Matrix

Cohorts defined by month of first subscription. Retention = MRR at offset N / MRR at offset 0.

- **Heatmap**: 35 cohorts × 35 month-offsets, red-to-green scale
- **Curve**: Average retention with P25–P75 band, truncated at M25 (min 10 cohorts)

In [8]:
COHORT_SQL = '''
WITH RECURSIVE months AS (
    SELECT '2023-01-01' AS month_start
    UNION ALL
    SELECT date(month_start, '+1 month') FROM months
    WHERE month_start < '2025-12-01'
),
user_cohort AS (
    SELECT user_id,
           strftime('%Y-%m', MIN(started_at)) AS cohort,
           strftime('%Y-%m-01', MIN(started_at)) AS cohort_date
    FROM fact_subscriptions
    GROUP BY user_id
),
sub_months AS (
    SELECT s.user_id, m.month_start, s.mrr
    FROM fact_subscriptions s
    JOIN months m
      ON m.month_start >= strftime('%Y-%m-01', s.started_at)
     AND (s.ended_at IS NULL
          OR m.month_start < strftime('%Y-%m-01', s.ended_at, '+1 month'))
),
customer_month AS (
    SELECT user_id, month_start, MAX(mrr) AS mrr
    FROM sub_months GROUP BY user_id, month_start
)
SELECT
    uc.cohort,
    (CAST(strftime('%Y', cm.month_start) AS INTEGER)
     - CAST(strftime('%Y', uc.cohort_date) AS INTEGER)) * 12
    + (CAST(strftime('%m', cm.month_start) AS INTEGER)
     - CAST(strftime('%m', uc.cohort_date) AS INTEGER)) AS month_offset,
    SUM(cm.mrr) AS active_mrr,
    COUNT(DISTINCT cm.user_id) AS active_users
FROM customer_month cm
JOIN user_cohort uc ON cm.user_id = uc.user_id
GROUP BY uc.cohort, month_offset
ORDER BY uc.cohort, month_offset;
'''

In [9]:
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA journal_mode=WAL')
print('Running cohort retention query...')
cohort_rows = conn.execute(COHORT_SQL).fetchall()
conn.close()
print(f'  {len(cohort_rows):,} cohort-month cells')

matrix = defaultdict(dict)
for cohort, offset, active_mrr, active_users in cohort_rows:
    matrix[cohort][offset] = active_mrr

cohorts = sorted(matrix.keys())
max_offset = max(offset for c in matrix.values() for offset in c.keys())

retention = {}
for cohort in cohorts:
    base_mrr = matrix[cohort].get(0, 0)
    if base_mrr == 0:
        continue
    retention[cohort] = {}
    for offset in range(max_offset + 1):
        if offset in matrix[cohort]:
            retention[cohort][offset] = round(
                100.0 * matrix[cohort][offset] / base_mrr, 1)

cohorts = sorted(retention.keys())
print(f'{len(cohorts)} cohorts, max offset M{max_offset}')

Running cohort retention query...
  630 cohort-month cells
35 cohorts, max offset M34


In [10]:
offsets = list(range(max_offset + 1))
with open(DATA_DIR / 'cohort_retention.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['cohort'] + [f'M{o}' for o in offsets])
    writer.writeheader()
    for cohort in cohorts:
        row = {'cohort': cohort}
        for o in offsets:
            row[f'M{o}'] = retention[cohort].get(o, '')
        writer.writerow(row)
print(f'Wrote {len(cohorts)} cohorts x {len(offsets)} months to cohort_retention.csv')

print(f"\n{'Cohort':>8s}", end='')
for o in range(min(13, len(offsets))):
    print(f'  M{o:>2d}', end='')
print()
for cohort in cohorts[:8]:
    print(f'{cohort:>8s}', end='')
    for o in range(min(13, len(offsets))):
        v = retention[cohort].get(o)
        print(f' {v:5.1f}' if v is not None else '     -', end='')
    print()

Wrote 35 cohorts x 35 months to cohort_retention.csv

  Cohort  M 0  M 1  M 2  M 3  M 4  M 5  M 6  M 7  M 8  M 9  M10  M11  M12
 2023-02 100.0 100.0 100.0 100.0 100.0 100.0 100.0 100.0 100.0 100.0 100.0  82.2  82.2
 2023-03 100.0 100.0  95.7  94.8  82.2  81.4  80.6  78.7  75.0  61.7  60.4  65.0  60.7
 2023-04 100.0 102.6  99.6  93.9  87.1  87.1  87.5  86.4  86.9  83.3  82.1  78.6  78.5
 2023-05 100.0 102.0 101.6  99.2  96.2  93.0  90.3  88.0  87.0  85.1  85.5  79.7  79.5
 2023-06 100.0 101.3  99.1  95.1  91.8  90.2  89.0  87.2  86.5  83.2  81.3  76.5  77.8
 2023-07 100.0 100.4  99.4  97.2  95.2  94.6  91.6  90.6  86.9  83.7  81.0  78.2  76.9
 2023-08 100.0 101.5  98.1  96.2  94.4  93.9  91.7  89.8  89.7  88.4  88.4  87.2  83.4
 2023-09 100.0 100.3  97.3  94.6  93.2  92.5  90.0  89.1  87.3  86.6  84.0  81.7  82.0


In [11]:
z, text = [], []
for cohort in cohorts:
    row_z = [retention[cohort].get(o) for o in offsets]
    z.append(row_z)
    text.append([f'{v:.0f}%' if v is not None else '' for v in row_z])

z_arr = np.array(z, dtype=float)
z_arr[z_arr == 0] = np.nan

fig_heatmap = go.Figure(data=go.Heatmap(
    z=z_arr,
    x=[f'M{o}' for o in offsets],
    y=cohorts,
    text=text,
    texttemplate='%{text}',
    textfont={'size': 8},
    colorscale=[
        [0.0,  '#d73027'],
        [0.3,  '#fc8d59'],
        [0.5,  '#fee08b'],
        [0.7,  '#d9ef8b'],
        [0.85, '#91cf60'],
        [1.0,  '#1a9850'],
    ],
    zmin=0, zmax=120,
    colorbar=dict(title='MRR<br>Retention %', ticksuffix='%'),
    hoverongaps=False,
    hovertemplate='Cohort: %{y}<br>Month: %{x}<br>Retention: %{z:.1f}%<extra></extra>',
))
fig_heatmap.update_layout(
    title='MRR Cohort Retention Heatmap',
    xaxis=dict(title='Months Since Signup', dtick=3, side='top'),
    yaxis=dict(title='Signup Cohort', autorange='reversed'),
    template='plotly_white',
    height=max(500, len(cohorts) * 22 + 120),
    width=max(900, len(offsets) * 30 + 120),
    margin=dict(l=80, r=100, t=80, b=40),
)
fig_heatmap.write_html(OUTPUT_DIR / 'cohort_retention.html', include_plotlyjs='cdn')
fig_heatmap.show()
fig_heatmap.write_image(OUTPUT_DIR / 'fig_heatmap.png')


### Static Image Preview\n
\n
![](../outputs/fig_heatmap.png)\n
\n

In [12]:
MIN_COHORTS = 10
avg_retention, cohort_counts = {}, {}
for o in offsets:
    values = [retention[c][o] for c in cohorts if o in retention[c]]
    cohort_counts[o] = len(values)
    if values:
        avg_retention[o] = round(np.mean(values), 1)

valid_offsets = sorted(o for o in avg_retention if cohort_counts[o] >= MIN_COHORTS)
max_plotted  = valid_offsets[-1] if valid_offsets else 0
avg_values   = [avg_retention[o] for o in valid_offsets]
n_values     = [cohort_counts[o]  for o in valid_offsets]
x_labels     = [f'M{o}' for o in valid_offsets]

p25 = [round(np.percentile(
           [retention[c][o] for c in cohorts if o in retention[c]], 25), 1)
       for o in valid_offsets]
p75 = [round(np.percentile(
           [retention[c][o] for c in cohorts if o in retention[c]], 75), 1)
       for o in valid_offsets]

fig_curve = go.Figure()
fig_curve.add_trace(go.Scatter(x=x_labels, y=p75, mode='lines',
                               line=dict(width=0), showlegend=False, hoverinfo='skip'))
fig_curve.add_trace(go.Scatter(x=x_labels, y=p25, mode='lines',
                               line=dict(width=0), fill='tonexty',
                               fillcolor='rgba(52,152,219,0.2)', name='P25-P75 range'))
fig_curve.add_trace(go.Scatter(
    x=x_labels, y=avg_values, mode='lines+markers', name='Average retention',
    line=dict(color='#2c3e50', width=2.5), marker=dict(size=5),
    hovertemplate='Month %{x}<br>Retention: %{y:.1f}%<br>n=%{customdata} cohorts<extra></extra>',
    customdata=n_values,
))
for i, o in enumerate(valid_offsets):
    fig_curve.add_annotation(x=x_labels[i], y=-0.08, yref='paper',
                             text=f'n={n_values[i]}', showarrow=False,
                             font=dict(size=8, color='#95a5a6'))
for target in [90, 80, 70]:
    for i, v in enumerate(avg_values):
        if v <= target:
            fig_curve.add_annotation(
                x=x_labels[i], y=v,
                text=f'{target}% at M{valid_offsets[i]}',
                showarrow=True, arrowhead=2, ax=40, ay=-30,
                font=dict(size=10, color='#7f8c8d'))
            break
fig_curve.add_annotation(
    text=f'Truncated after M{max_plotted} (minimum {MIN_COHORTS} cohorts required). '
         f'M{max_plotted+1}-M{max(offsets)} excluded due to small sample sizes.',
    xref='paper', yref='paper', x=0.5, y=-0.18, showarrow=False,
    font=dict(size=10, color='#7f8c8d', style='italic'), align='center',
)
fig_curve.update_layout(
    title='Average MRR Retention Curve (All Cohorts)',
    xaxis=dict(title='Months Since Signup', dtick=3),
    yaxis=dict(title='MRR Retention %', ticksuffix='%',
               range=[0, max(avg_values) * 1.05]),
    template='plotly_white', height=500, width=900,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    margin=dict(l=80, r=40, t=80, b=100),
)
fig_curve.write_html(OUTPUT_DIR / 'cohort_retention_curve.html', include_plotlyjs='cdn')
fig_curve.show()
fig_curve.write_image(OUTPUT_DIR / 'fig_curve.png')

print('Key metrics:')
for mo in [1, 6, 12, 24]:
    if mo in avg_retention:
        print(f'  M{mo:>2d} avg retention: {avg_retention[mo]}%')

Key metrics:
  M 1 avg retention: 101.5%
  M 6 avg retention: 91.2%
  M12 avg retention: 81.2%
  M24 avg retention: 71.6%


### Static Image Preview\n
\n
![](../outputs/fig_curve.png)\n
\n

## 4. Retention by Channel + Anomaly

Two views of the same data designed to surface a seeded anomaly:

1. **Logo retention (all plans)** — weights all plans equally; `partner_referral` blends in (anomaly hidden)
2. **MRR retention (Enterprise only)** — exposes `partner_referral`'s 3× Enterprise churn rate

In [13]:
MIN_COHORTS_CH = 8

def build_channel_query(plan_filter=None):
    plan_clause = f"AND s.plan = '{plan_filter}'" if plan_filter else ''
    return f'''
    WITH RECURSIVE months AS (
        SELECT '2023-01-01' AS month_start
        UNION ALL
        SELECT date(month_start, '+1 month') FROM months
        WHERE month_start < '2025-12-01'
    ),
    user_cohort AS (
        SELECT s.user_id,
               u.acquisition_channel AS channel,
               strftime('%Y-%m', MIN(s.started_at)) AS cohort,
               strftime('%Y-%m-01', MIN(s.started_at)) AS cohort_date
        FROM fact_subscriptions s
        JOIN dim_users u ON s.user_id = u.user_id
        WHERE 1=1 {plan_clause}
        GROUP BY s.user_id
    ),
    sub_months AS (
        SELECT s.user_id, m.month_start, s.mrr
        FROM fact_subscriptions s
        JOIN months m
          ON m.month_start >= strftime('%Y-%m-01', s.started_at)
         AND (s.ended_at IS NULL
              OR m.month_start < strftime('%Y-%m-01', s.ended_at, '+1 month'))
        WHERE 1=1 {plan_clause}
    ),
    customer_month AS (
        SELECT user_id, month_start, MAX(mrr) AS mrr
        FROM sub_months GROUP BY user_id, month_start
    )
    SELECT
        uc.channel, uc.cohort,
        (CAST(strftime('%Y', cm.month_start) AS INTEGER)
         - CAST(strftime('%Y', uc.cohort_date) AS INTEGER)) * 12
        + (CAST(strftime('%m', cm.month_start) AS INTEGER)
         - CAST(strftime('%m', uc.cohort_date) AS INTEGER)) AS month_offset,
        SUM(cm.mrr) AS active_mrr,
        COUNT(DISTINCT cm.user_id) AS active_users
    FROM customer_month cm
    JOIN user_cohort uc ON cm.user_id = uc.user_id
    GROUP BY uc.channel, uc.cohort, month_offset
    ORDER BY uc.channel, uc.cohort, month_offset;
    '''

def compute_channel_retention(rows, mode='mrr'):
    raw = defaultdict(lambda: defaultdict(dict))
    val_idx = 3 if mode == 'mrr' else 4
    for row in rows:
        channel, cohort, offset = row[0], row[1], row[2]
        raw[channel][cohort][offset] = row[val_idx]
    result = {}
    for channel, cohort_data in raw.items():
        ret_by_cohort = {}
        for cohort, offs_d in cohort_data.items():
            base = offs_d.get(0, 0)
            if base == 0:
                continue
            ret_by_cohort[cohort] = {o: 100.0 * offs_d[o] / base for o in offs_d}
        max_off = max(o for r in ret_by_cohort.values() for o in r)
        curve = {}
        for o in range(max_off + 1):
            vals = [r[o] for r in ret_by_cohort.values() if o in r]
            if len(vals) >= MIN_COHORTS_CH:
                curve[o] = (round(np.mean(vals), 1), len(vals))
        result[channel] = curve
    return result

def plot_channel_curves(retention_data, title, subtitle, output_path,
                         highlight_channel=None):
    fig = go.Figure()
    for channel in sorted(retention_data.keys()):
        curve = retention_data[channel]
        if not curve:
            continue
        offs = sorted(curve.keys())
        vals   = [curve[o][0] for o in offs]
        n_vals = [curve[o][1] for o in offs]
        x_lbl  = [f'M{o}' for o in offs]
        is_hl  = (channel == highlight_channel)
        fig.add_trace(go.Scatter(
            x=x_lbl, y=vals, mode='lines+markers', name=channel,
            line=dict(color=CHANNEL_COLORS.get(channel, '#999'),
                      width=3.5 if is_hl else 2, dash='solid'),
            marker=dict(size=6 if is_hl else 4),
            opacity=1.0 if is_hl else 0.7,
            customdata=list(zip([channel] * len(offs), n_vals)),
            hovertemplate=('%{customdata[0]}<br>Month %{x}<br>'
                           'Retention: %{y:.1f}%<br>n=%{customdata[1]}<extra></extra>'),
        ))
        last_o, last_v = max(curve.keys()), curve[max(curve.keys())][0]
        fig.add_annotation(x=f'M{last_o}', y=last_v,
                           text=f' {channel} ({last_v:.0f}%)',
                           showarrow=False, xanchor='left', xshift=8,
                           font=dict(size=9, color=CHANNEL_COLORS.get(channel, '#999')))
    fig.update_layout(
        title=dict(text=f'{title}<br><sup>{subtitle}</sup>'),
        xaxis=dict(title='Months Since Signup', dtick=3),
        yaxis=dict(title='Retention %', ticksuffix='%', range=[0, 115]),
        template='plotly_white', height=550, width=1000,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        margin=dict(l=80, r=150, t=100, b=80),
    )
    fig.add_annotation(
        text=f'Only showing months where >= {MIN_COHORTS_CH} cohorts contribute data per channel.',
        xref='paper', yref='paper', x=0.5, y=-0.13, showarrow=False,
        font=dict(size=10, color='#7f8c8d', style='italic'),
    )
    fig.write_html(output_path, include_plotlyjs='cdn')
    return fig

In [14]:
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA journal_mode=WAL')
print('Running all-plans logo retention by channel...')
rows_all = conn.execute(build_channel_query(plan_filter=None)).fetchall()
print(f'  {len(rows_all):,} rows')
ret_all = compute_channel_retention(rows_all, mode='logo')

for ch, curve in sorted(ret_all.items()):
    m12 = curve.get(12, (None, 0))
    print(f'  {ch:22s}  M12={m12[0]}%' if m12[0] else f'  {ch:22s}  M12=N/A')

fig_all = plot_channel_curves(
    ret_all,
    title='Logo Retention by Acquisition Channel (All Plans)',
    subtitle='partner_referral blends in — anomaly is hidden at the logo level',
    output_path=str(OUTPUT_DIR / 'cohort_retention_by_channel.html'),
    highlight_channel='partner_referral',
)
fig_all.show()
fig_all.write_image(OUTPUT_DIR / 'fig_all.png')


Running all-plans logo retention by channel...
  3,646 rows
  content_marketing       M12=70.4%
  direct                  M12=73.6%
  organic_search          M12=71.3%
  paid_search             M12=69.3%
  partner_referral        M12=66.3%
  social_media            M12=69.5%


### Static Image Preview\n
\n
![](../outputs/fig_all.png)\n
\n

In [15]:
print('Running Enterprise-only MRR retention by channel...')
rows_ent = conn.execute(build_channel_query(plan_filter='Enterprise')).fetchall()
print(f'  {len(rows_ent):,} rows')
ret_ent = compute_channel_retention(rows_ent, mode='mrr')
conn.close()

for ch, curve in sorted(ret_ent.items()):
    m12 = curve.get(12, (None, 0))
    anomaly = ' <-- ANOMALY' if ch == 'partner_referral' else ''
    print(f'  {ch:22s}  M12={m12[0]}%{anomaly}' if m12[0] else
          f'  {ch:22s}  M12=N/A')

fig_ent = plot_channel_curves(
    ret_ent,
    title='MRR Retention by Acquisition Channel (Enterprise Only)',
    subtitle='partner_referral anomaly exposed — significantly worse retention than other channels',
    output_path=str(OUTPUT_DIR / 'cohort_retention_by_channel_enterprise.html'),
    highlight_channel='partner_referral',
)
fig_ent.show()
fig_ent.write_image(OUTPUT_DIR / 'fig_ent.png')

print('\n-- Anomaly comparison at M12 --')
print(f"{'Channel':22s} {'Logo(All)':>10s} {'MRR(Ent)':>12s}")
for ch in sorted(CHANNEL_COLORS.keys()):
    all_m12 = ret_all.get(ch, {}).get(12, (None, 0))[0]
    ent_m12 = ret_ent.get(ch, {}).get(12, (None, 0))[0]
    if all_m12 and ent_m12:
        marker = ' <-- ANOMALY' if ch == 'partner_referral' else ''
        print(f'{ch:22s} {all_m12:>9.1f}% {ent_m12:>11.1f}%{marker}')

Running Enterprise-only MRR retention by channel...
  3,416 rows
  content_marketing       M12=78.3%
  direct                  M12=75.0%
  organic_search          M12=78.4%
  paid_search             M12=82.5%
  partner_referral        M12=58.6% <-- ANOMALY
  social_media            M12=79.5%



-- Anomaly comparison at M12 --
Channel                 Logo(All)     MRR(Ent)
content_marketing           70.4%        78.3%
direct                      73.6%        75.0%
organic_search              71.3%        78.4%
paid_search                 69.3%        82.5%
partner_referral            66.3%        58.6% <-- ANOMALY
social_media                69.5%        79.5%


### Static Image Preview\n
\n
![](../outputs/fig_ent.png)\n
\n

## 5. LTV/CAC Analysis

LTV = avg MRR × (1 / observed monthly churn rate), computed per channel × plan tier.

- **Bubble chart**: x = payback months, y = LTV:CAC ratio, size = MRR volume
- **Grouped bar chart**: LTV by channel × plan, with reference line at avg Enterprise LTV (excl. `partner_referral`)

In [16]:
CAC_BY_CHANNEL = {
    'organic_search':    120,
    'paid_search':       340,
    'social_media':      280,
    'content_marketing': 150,
    'partner_referral':  95,
    'direct':            60,
}

SEGMENT_SQL = '''
WITH first_sub AS (
    SELECT s.user_id, s.plan, s.mrr, u.acquisition_channel,
           ROW_NUMBER() OVER (PARTITION BY s.user_id ORDER BY s.started_at) AS rn
    FROM fact_subscriptions s
    JOIN dim_users u ON s.user_id = u.user_id
),
sub_life AS (
    SELECT s.user_id, s.plan, u.acquisition_channel,
           CAST((JULIANDAY(COALESCE(s.ended_at, '2025-12-31'))
                 - JULIANDAY(s.started_at)) / 30.44 AS INTEGER) AS months_active,
           CASE WHEN s.end_reason = 'churned' THEN 1 ELSE 0 END AS did_churn
    FROM fact_subscriptions s
    JOIN dim_users u ON s.user_id = u.user_id
    WHERE JULIANDAY(COALESCE(s.ended_at, '2025-12-31')) > JULIANDAY(s.started_at)
)
SELECT
    f.acquisition_channel AS channel,
    f.plan,
    COUNT(DISTINCT f.user_id) AS users,
    ROUND(AVG(f.mrr), 2) AS avg_initial_mrr,
    ROUND(SUM(f.mrr), 0) AS total_initial_mrr,
    sl.total_months,
    sl.total_churns,
    sl.monthly_churn_rate
FROM first_sub f
JOIN (
    SELECT acquisition_channel, plan,
           SUM(months_active) AS total_months,
           SUM(did_churn) AS total_churns,
           ROUND(1.0 * SUM(did_churn) / SUM(months_active), 6) AS monthly_churn_rate
    FROM sub_life
    GROUP BY acquisition_channel, plan
) sl ON f.acquisition_channel = sl.acquisition_channel AND f.plan = sl.plan
WHERE f.rn = 1
GROUP BY f.acquisition_channel, f.plan
ORDER BY f.acquisition_channel, f.plan;
'''

In [17]:
conn = sqlite3.connect(DB_PATH)
ltv_rows = conn.execute(SEGMENT_SQL).fetchall()
conn.close()

segments = []
for channel, plan, users, avg_mrr, total_mrr, tot_months, churns, churn_rate in ltv_rows:
    cac = CAC_BY_CHANNEL[channel]
    avg_lifetime = 1.0 / churn_rate if churn_rate > 0 else 999
    ltv = avg_mrr * avg_lifetime
    segments.append({
        'channel': channel, 'plan': plan, 'users': users,
        'avg_mrr': round(avg_mrr, 2),
        'total_initial_mrr': total_mrr,
        'monthly_churn_pct': round(churn_rate * 100, 2),
        'avg_lifetime_months': round(avg_lifetime, 1),
        'ltv': round(ltv, 0),
        'cac': cac,
        'ltv_cac_ratio': round(ltv / cac, 1),
        'payback_months': round(cac / avg_mrr if avg_mrr > 0 else 999, 1),
    })

channel_agg = {}
for s in segments:
    ch = s['channel']
    if ch not in channel_agg:
        channel_agg[ch] = {'total_users': 0, 'weighted_ltv': 0.0,
                            'total_mrr': 0, 'cac': s['cac']}
    channel_agg[ch]['total_users'] += s['users']
    channel_agg[ch]['weighted_ltv'] += s['ltv'] * s['users']
    channel_agg[ch]['total_mrr'] += s['total_initial_mrr']

blended = []
for ch, agg in channel_agg.items():
    blended_ltv = agg['weighted_ltv'] / agg['total_users']
    avg_mrr_b   = agg['total_mrr'] / agg['total_users']
    blended.append({
        'channel': ch, 'users': agg['total_users'],
        'blended_ltv': round(blended_ltv, 0), 'cac': agg['cac'],
        'ltv_cac_ratio': round(blended_ltv / agg['cac'], 1),
        'payback_months': round(agg['cac'] / avg_mrr_b, 1),
        'total_mrr': agg['total_mrr'],
    })
print(f'{len(segments)} segments computed')

18 segments computed


In [18]:
with open(DATA_DIR / 'ltv_cac.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(segments[0].keys()))
    writer.writeheader()
    writer.writerows(segments)
print(f'Wrote {len(segments)} segments to ltv_cac.csv')

print(f"\n{'Channel':22s} {'Plan':12s} {'Users':>6s} {'MRR':>7s} {'Churn%':>7s} "
      f"{'Life(mo)':>8s} {'LTV':>8s} {'CAC':>5s} {'LTV:CAC':>8s} {'Payback':>8s}")
for s in segments:
    marker = ' **' if s['channel'] == 'partner_referral' and s['plan'] == 'Enterprise' else ''
    print(f"{s['channel']:22s} {s['plan']:12s} {s['users']:>6d} "
          f"${s['avg_mrr']:>5.0f} {s['monthly_churn_pct']:>6.2f}% "
          f"{s['avg_lifetime_months']:>7.1f} ${s['ltv']:>6,.0f} "
          f"${s['cac']:>4d} {s['ltv_cac_ratio']:>7.1f}x "
          f"{s['payback_months']:>6.1f}mo{marker}")

Wrote 18 segments to ltv_cac.csv

Channel                Plan          Users     MRR  Churn% Life(mo)      LTV   CAC  LTV:CAC  Payback
content_marketing      Enterprise      214 $  281   1.69%    59.2 $16,651 $ 150   111.0x    0.5mo
content_marketing      Growth          475 $   93   3.09%    32.4 $ 3,012 $ 150    20.1x    1.6mo
content_marketing      Starter         820 $   27   4.56%    21.9 $   598 $ 150     4.0x    5.5mo
direct                 Enterprise      120 $  279   2.08%    48.1 $13,451 $  60   224.2x    0.2mo
direct                 Growth          333 $   93   3.41%    29.3 $ 2,737 $  60    45.6x    0.6mo
direct                 Starter         537 $   27   4.79%    20.9 $   570 $  60     9.5x    2.2mo
organic_search         Enterprise      377 $  279   1.93%    51.9 $14,494 $ 120   120.8x    0.4mo
organic_search         Growth          968 $   93   3.06%    32.7 $ 3,045 $ 120    25.4x    1.3mo
organic_search         Starter        1721 $   27   4.77%    21.0 $   575 $ 120  

In [19]:
fig_bubble = go.Figure()
for plan in ['Starter', 'Growth', 'Enterprise']:
    plan_segs = [s for s in segments if s['plan'] == plan]
    fig_bubble.add_trace(go.Scatter(
        x=[s['payback_months'] for s in plan_segs],
        y=[s['ltv_cac_ratio']  for s in plan_segs],
        mode='markers', name=plan,
        marker=dict(
            size=[max(8, s['total_initial_mrr'] ** 0.5 / 2) for s in plan_segs],
            color=[CHANNEL_COLORS[s['channel']] for s in plan_segs],
            opacity=0.75, line=dict(width=1.5, color='white'),
        ),
        text=[f"{s['channel']}<br>{s['plan']}<br>LTV: ${s['ltv']:,.0f}<br>"
              f"CAC: ${s['cac']}<br>LTV:CAC: {s['ltv_cac_ratio']}x<br>"
              f"Payback: {s['payback_months']}mo<br>Churn: {s['monthly_churn_pct']}%/mo"
              for s in plan_segs],
        hoverinfo='text',
        marker_symbol={'Starter': 'circle', 'Growth': 'diamond', 'Enterprise': 'square'}[plan],
    ))

pr_ent       = next(s for s in segments
                    if s['channel'] == 'partner_referral' and s['plan'] == 'Enterprise')
other_ent    = [s for s in segments
                if s['plan'] == 'Enterprise' and s['channel'] != 'partner_referral']
avg_other_ent_ltv = round(sum(s['ltv'] for s in other_ent) / len(other_ent), 0)
best_ent     = max(other_ent, key=lambda s: s['ltv_cac_ratio'])

fig_bubble.add_annotation(
    x=pr_ent['payback_months'], y=pr_ent['ltv_cac_ratio'],
    text=f"partner_referral Enterprise<br>LTV:CAC {pr_ent['ltv_cac_ratio']}x | "
         f"Churn {pr_ent['monthly_churn_pct']}%/mo<br><i>3x normal Enterprise churn</i>",
    showarrow=True, arrowhead=2, ax=80, ay=-60,
    bordercolor='#e74c3c', borderwidth=1.5, bgcolor='rgba(231,76,60,0.08)',
    font=dict(size=10),
)
fig_bubble.add_annotation(
    x=best_ent['payback_months'], y=best_ent['ltv_cac_ratio'],
    text=f"{best_ent['channel']} Enterprise<br>LTV:CAC {best_ent['ltv_cac_ratio']}x | "
         f"Churn {best_ent['monthly_churn_pct']}%/mo",
    showarrow=True, arrowhead=2, ax=-80, ay=-40,
    bordercolor='#27ae60', borderwidth=1.5, bgcolor='rgba(39,174,96,0.08)',
    font=dict(size=10),
)
legend_y = 0.98
for ch, color in sorted(CHANNEL_COLORS.items()):
    fig_bubble.add_annotation(
        text=f'<span style="color:{color};">&#9679;</span> {ch}',
        xref='paper', yref='paper', x=1.02, y=legend_y,
        showarrow=False, xanchor='left', font=dict(size=10),
    )
    legend_y -= 0.045
fig_bubble.add_hline(y=3.0, line_dash='dot', line_color='#bdc3c7',
                     annotation_text='LTV:CAC = 3x (healthy threshold)',
                     annotation_position='top left',
                     annotation_font=dict(size=9, color='#95a5a6'))
fig_bubble.update_layout(
    title=dict(text='LTV:CAC Analysis by Channel and Plan Tier<br>'
                    '<sup>Bubble size = initial MRR volume | '
                    'Shape: circle=Starter, diamond=Growth, square=Enterprise</sup>'),
    xaxis=dict(title='CAC Payback Period (months)', dtick=1),
    yaxis=dict(title='LTV:CAC Ratio', ticksuffix='x'),
    template='plotly_white', height=600, width=1050,
    margin=dict(l=80, r=180, t=100, b=80), showlegend=True,
    legend=dict(title='Plan Tier', orientation='h',
                yanchor='bottom', y=0.98, xanchor='center', x=0.4),
)
fig_bubble.write_html(OUTPUT_DIR / 'ltv_cac.html', include_plotlyjs='cdn')
fig_bubble.show()
fig_bubble.write_image(OUTPUT_DIR / 'fig_bubble.png')


### Static Image Preview\n
\n
![](../outputs/fig_bubble.png)\n
\n

In [20]:
channels_ordered = sorted({s['channel'] for s in segments})

fig_grouped = go.Figure()
for plan in ['Starter', 'Growth', 'Enterprise']:
    plan_segs = {s['channel']: s for s in segments if s['plan'] == plan}
    ltv_vals  = [plan_segs[ch]['ltv'] if ch in plan_segs else 0
                 for ch in channels_ordered]
    fig_grouped.add_trace(go.Bar(
        name=plan, x=channels_ordered, y=ltv_vals,
        marker_color=PLAN_COLORS[plan],
        hovertemplate=f'<b>%{{x}}</b><br>{plan}<br>LTV: $%{{y:,.0f}}<extra></extra>',
    ))

fig_grouped.add_hline(
    y=avg_other_ent_ltv, line_dash='dash', line_color='#888', line_width=1.5,
    annotation_text=f'Avg Enterprise LTV (excl. partner_referral): ${avg_other_ent_ltv:,.0f}',
    annotation_position='top right',
    annotation_font=dict(size=10, color='#555'),
)
pr_idx = channels_ordered.index('partner_referral')
fig_grouped.add_annotation(
    x=pr_idx, y=pr_ent['ltv'], xref='x', yref='y',
    text=f"${pr_ent['ltv']:,.0f}<br><i>vs ${avg_other_ent_ltv:,.0f} avg</i>",
    showarrow=True, arrowhead=2, ax=0, ay=-50,
    font=dict(size=10, color='#c0392b'),
    bordercolor='#e74c3c', borderwidth=1.5, bgcolor='rgba(231,76,60,0.08)',
)
fig_grouped.update_layout(
    barmode='group',
    title=dict(
        text='LTV by Acquisition Channel and Plan Tier<br>'
             '<sup>Dashed line = avg Enterprise LTV across non-partner_referral channels</sup>',
    ),
    xaxis=dict(title='Acquisition Channel', tickangle=-20),
    yaxis=dict(title='LTV ($)', tickprefix='$', separatethousands=True),
    template='plotly_white', height=550, width=1000,
    margin=dict(l=80, r=60, t=100, b=100),
    legend=dict(title='Plan Tier', orientation='h',
                yanchor='bottom', y=0.95, xanchor='center', x=0.5),
)
fig_grouped.write_html(OUTPUT_DIR / 'ltv_cac_grouped.html', include_plotlyjs='cdn')
fig_grouped.show()
fig_grouped.write_image(OUTPUT_DIR / 'fig_grouped.png')


### Static Image Preview\n
\n
![](../outputs/fig_grouped.png)\n
\n

In [21]:
pr_blended = next(b for b in blended if b['channel'] == 'partner_referral')
print('-- partner_referral Enterprise damage assessment --')
print(f"  partner_referral Enterprise LTV:   ${pr_ent['ltv']:,.0f}")
print(f'  Other channels Enterprise avg LTV:  ${avg_other_ent_ltv:,.0f}')
print(f"  LTV destruction:  ${avg_other_ent_ltv - pr_ent['ltv']:,.0f} "
      f"({100 * (1 - pr_ent['ltv'] / avg_other_ent_ltv):.0f}% lower)")
print(f"  partner_referral Enterprise LTV:CAC: {pr_ent['ltv_cac_ratio']}x")
print(f"  partner_referral blended   LTV:CAC:  {pr_blended['ltv_cac_ratio']}x "
      f'(masks the problem because Starter/Growth are fine)')

-- partner_referral Enterprise damage assessment --
  partner_referral Enterprise LTV:   $4,654
  Other channels Enterprise avg LTV:  $14,567
  LTV destruction:  $9,913 (68% lower)
  partner_referral Enterprise LTV:CAC: 49.0x
  partner_referral blended   LTV:CAC:  20.3x (masks the problem because Starter/Growth are fine)


## 6. Interactive Dashboard

The `saas_dashboard.py` Dash app bundles all four analyses into a single interactive dashboard with a tab-aware filter sidebar.

### App structure

| Component | Details |
|---|---|
| **Sidebar** | Date-range slider (2023-01 – 2025-12), Plan checklist, Channel checklist |
| **Tab 1 — MRR Waterfall** | Stacked bar + Ending MRR line; date-range filter applies |
| **Tab 2 — Cohort Retention** | Annotated heatmap + average curve; date-range filter selects which cohorts are shown |
| **Tab 3 — Retention by Channel** | Logo retention (all plans) or MRR retention (Enterprise only, toggled via Plan filter); channel filter hides/shows individual channels |
| **Tab 4 — LTV / CAC** | Bubble chart + grouped bar chart; plan and channel filters apply |

**Filter scope:**
- *Date range* → MRR Waterfall, Cohort Retention
- *Plan* → Retention by Channel (toggles all-plans vs Enterprise view), LTV/CAC
- *Channel* → Retention by Channel, LTV/CAC

Data is pre-loaded from the CSVs and DB at startup; callbacks do in-memory filtering so the UI stays responsive.

Run standalone: `python saas_dashboard.py` → [http://localhost:8050](http://localhost:8050)

**Run mode:** launches in `jupyter_mode="external"` on port `8050` for stable notebook execution.


In [22]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
BASE_DIR = CWD if (CWD / 'dashboard').exists() else CWD.parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from dashboard.saas_dashboard import app

app.run(jupyter_mode="external", debug=False, use_reloader=False, port=8050)


Dash app running on http://127.0.0.1:8050/
